# 🏁 RaceIntel — Notebook

Construcción paso a paso del agente **RaceIntel** con LangGraph y trazabilidad en LangSmith.

**Workflow:** `START → race_agent ⇄ tools → END`  
El agente sigue el patrón **ReAct**: el LLM decide qué herramienta llamar (y cuántas veces) antes de dar una respuesta final.

---
## Índice
1. Instalación de dependencias
2. Configuración de credenciales
3. Estado del grafo (`RaceIntelState`)
4. Herramientas de búsqueda
5. Nodo agente (`race_agent`)
6. Construcción del grafo
7. Visualización del grafo
8. Ejecución — consultas de ejemplo


## 1. Instalación de dependencias

In [ ]:
%pip install -q langgraph langchain langchain-openai langchain-community \
               langsmith tavily-python python-dotenv wikipedia


## 2. Configuración de credenciales

Carga las variables desde un archivo `.env` en el mismo directorio que el notebook.  
El archivo debe contener:
```
OPENAI_API_KEY=...
AZURE_OPENAI_BASE_URL=...
TAVILY_API_KEY=...
LANGSMITH_TRACING=true
LANGSMITH_API_KEY=...
LANGSMITH_PROJECT=raceintel
```


In [ ]:
import os
from dotenv import load_dotenv

load_dotenv()  # Carga .env desde el directorio actual

# Verifica que las variables críticas estén presentes
required = ["OPENAI_API_KEY", "AZURE_OPENAI_BASE_URL", "TAVILY_API_KEY", "LANGSMITH_API_KEY"]
missing = [v for v in required if not os.getenv(v)]

if missing:
    print(f"⚠️  Variables no encontradas: {missing}")
else:
    print("✅ Todas las credenciales cargadas correctamente")
    print(f"   LangSmith project: {os.getenv('LANGSMITH_PROJECT', 'default')}")
    print(f"   Tracing activo: {os.getenv('LANGSMITH_TRACING', 'false')}")


## 3. Estado del grafo — `RaceIntelState`

El estado es un `TypedDict` con dos campos:
- **`messages`**: historial completo. Usa `add_messages` como reducer → los mensajes se *acumulan* en lugar de sobrescribirse.
- **`session_id`**: identifica la sesión en LangSmith.


In [ ]:
from typing import Annotated
from typing_extensions import TypedDict
from langgraph.graph.message import add_messages


class RaceIntelState(TypedDict):
    """Estado compartido del grafo RaceIntel.

    - messages: historial completo (HumanMessage, AIMessage, ToolMessage).
      add_messages acumula sin sobrescribir.
    - session_id: identificador de sesión para LangSmith tracing.
    """
    messages: Annotated[list, add_messages]
    session_id: str


print("✅ RaceIntelState definido")


## 4. Herramientas de búsqueda

Cuatro herramientas Tavily, cada una enfocada en un tipo de consulta diferente.
El decorador `@tool` expone la docstring al LLM para que sepa cuándo usarla.

| Herramienta | Cuándo se activa |
|---|---|
| `search_race_results` | Ganadores, podios, tiempos |
| `search_driver_stats` | Estadísticas y palmarés de pilotos |
| `search_championship_standings` | Clasificaciones de campeonato |
| `search_race_calendar` | Calendario y próximas carreras |


In [ ]:
from langchain_core.tools import tool
from langchain_community.tools.tavily_search import TavilySearchResults


def _tavily(max_results: int = 4) -> TavilySearchResults:
    """Instancia un cliente Tavily con el número de resultados indicado."""
    return TavilySearchResults(
        max_results=max_results,
        tavily_api_key=os.getenv("TAVILY_API_KEY"),
    )


@tool
def search_race_results(query: str) -> str:
    """Busca resultados de carreras: ganadores, podios y tiempos de vuelta.

    Úsala para preguntas como: '¿quién ganó el GP de Mónaco 2024?',
    '¿cuál fue el podio en Le Mans 2023?', 'resultado Dakar etapa 5 2025'.
    """
    refined = f"race result winner podium {query}"
    results = _tavily(4).invoke(refined)
    if not results:
        return "No se encontraron resultados para esa carrera."
    return "\n\n".join(
        f"[{r.get('url', '')}]\n{r.get('content', '')[:300]}"
        for r in results if isinstance(r, dict)
    )


@tool
def search_driver_stats(query: str) -> str:
    """Busca estadísticas, palmarés y biografía de pilotos.

    Úsala para preguntas como: '¿cuántos mundiales tiene Verstappen?',
    'estadísticas de Fernando Alonso en F1', 'historial de Scott Dixon en IndyCar'.
    """
    refined = f"driver statistics career stats {query}"
    results = _tavily(4).invoke(refined)
    if not results:
        return "No se encontraron estadísticas para ese piloto."
    return "\n\n".join(
        f"[{r.get('url', '')}]\n{r.get('content', '')[:300]}"
        for r in results if isinstance(r, dict)
    )


@tool
def search_championship_standings(query: str) -> str:
    """Busca clasificaciones de campeonatos: F1, IndyCar, NASCAR, WEC, Dakar, etc.

    Úsala para preguntas como: '¿cómo va el campeonato de F1 2025?',
    'clasificación WEC 2024 LMP1', 'standings NASCAR Cup Series'.
    """
    refined = f"championship standings classification 2025 {query}"
    results = _tavily(4).invoke(refined)
    if not results:
        return "No se encontraron datos de clasificación."
    return "\n\n".join(
        f"[{r.get('url', '')}]\n{r.get('content', '')[:300]}"
        for r in results if isinstance(r, dict)
    )


@tool
def search_race_calendar(query: str) -> str:
    """Busca calendarios de temporada y próximas citas de carreras.

    Úsala para preguntas como: '¿cuándo es la próxima carrera de F1?',
    'calendario IndyCar 2025', '¿qué circuitos tiene el WEC esta temporada?'.
    """
    refined = f"race calendar schedule next race 2025 {query}"
    results = _tavily(3).invoke(refined)
    if not results:
        return "No se encontró información de calendario."
    return "\n\n".join(
        f"[{r.get('url', '')}]\n{r.get('content', '')[:300]}"
        for r in results if isinstance(r, dict)
    )


TOOLS = [search_race_results, search_driver_stats, search_championship_standings, search_race_calendar]
print(f"✅ {len(TOOLS)} herramientas definidas: {[t.name for t in TOOLS]}")


## 5. Nodo agente — `race_agent`

El LLM recibe el historial completo más un `SystemMessage` con las instrucciones.  
`bind_tools` adjunta las herramientas al modelo: si el LLM quiere usarlas, devuelve un `AIMessage` con `tool_calls`; si no, devuelve la respuesta final.


In [ ]:
from langchain_openai import ChatOpenAI
from langchain_core.messages import SystemMessage


SYSTEM_PROMPT = """Eres RaceIntel, un asistente especializado en automovilismo.
Cubres F1, IndyCar, NASCAR, WEC (Le Mans), Dakar y otras series de velocidad.

Reglas:
- Usa las herramientas disponibles para obtener datos actualizados antes de responder.
- Responde en español con tono conciso y periodístico (máx. 3 párrafos).
- Cita las fuentes cuando sea relevante.
- Si la pregunta no es sobre automovilismo, indícalo amablemente.
- Nunca inventes datos; si no encuentras información, dilo."""


# Construye el LLM con las herramientas enlazadas
_llm = ChatOpenAI(
    api_key=os.getenv("OPENAI_API_KEY"),
    base_url=os.getenv("AZURE_OPENAI_BASE_URL"),
    model="gpt-4o-mini",
    temperature=0.2,
).bind_tools(TOOLS)


def race_agent(state: RaceIntelState) -> RaceIntelState:
    """Invoca el LLM con el historial de mensajes y devuelve la respuesta."""
    messages = [SystemMessage(content=SYSTEM_PROMPT)] + state["messages"]
    response = _llm.invoke(messages)
    print(
        f"🏎️  [RaceAgent] "
        + ("Llamando herramientas: " + str([tc['name'] for tc in response.tool_calls])
           if response.tool_calls else "Respuesta final lista")
    )
    return {"messages": [response]}


print("✅ race_agent definido")


## 6. Construcción del grafo

```
START
  │
  ▼
race_agent ──── tool_calls? ──YES──► tools
  ▲                                    │
  └────────────────────────────────────┘
  │
  NO
  │
  ▼
 END
```

`tools_condition` es una arista condicional de LangGraph que lee `tool_calls` del último mensaje: si hay llamadas, redirige al nodo `tools`; si no, va a `END`.


In [ ]:
from langgraph.graph import StateGraph, START, END
from langgraph.prebuilt import ToolNode, tools_condition


def build_graph():
    """Construye el grafo principal y lo devuelve compilado."""
    tool_node = ToolNode(TOOLS)

    builder = StateGraph(RaceIntelState)

    # Nodos
    builder.add_node("race_agent", race_agent)
    builder.add_node("tools", tool_node)

    # Aristas
    builder.add_edge(START, "race_agent")
    builder.add_conditional_edges(
        "race_agent",
        tools_condition,  # → "tools" si hay tool_calls, → END si no
    )
    builder.add_edge("tools", "race_agent")  # resultado vuelve al agente

    return builder.compile()


graph = build_graph()
print("✅ Grafo compilado")


## 7. Visualización del grafo

LangGraph puede renderizar el grafo como imagen PNG usando `draw_mermaid_png()`.
Requiere `pygraphviz` o acceso a Mermaid. Si no está disponible, se muestra la representación ASCII.


In [ ]:
from IPython.display import Image, display

try:
    img = graph.get_graph().draw_mermaid_png()
    display(Image(img))
except Exception as e:
    # Fallback: representación en texto
    print("⚠️  No se pudo renderizar como imagen:", e)
    print()
    print(graph.get_graph().draw_ascii())


## 8. Ejecución — función de consulta

La función `ask()` encapsula una invocación completa del grafo para una sola pregunta.
Cada llamada genera un `run_name` único visible en LangSmith.


In [ ]:
import uuid
from langchain_core.messages import HumanMessage

SESSION_ID = str(uuid.uuid4())[:8]  # ID fijo para toda la sesión del notebook
print(f"🔗 Session ID: {SESSION_ID}")


def ask(question: str) -> str:
    """Lanza una consulta al agente y devuelve la respuesta final."""
    print(f"\n{'='*60}")
    print(f"❓ Pregunta: {question}")
    print(f"{'='*60}")

    result = graph.invoke(
        {
            "messages": [HumanMessage(content=question)],
            "session_id": SESSION_ID,
        },
        config={
            "configurable": {"thread_id": SESSION_ID},
            "run_name": f"raceintel-{SESSION_ID}",
        },
    )

    answer = result["messages"][-1].content
    print(f"\n🏎️  RaceIntel:\n{answer}")
    return answer


### 8.1 Resultados de carrera

In [ ]:
ask("¿Quién ganó el GP de Mónaco de F1 2024?")


### 8.2 Estadísticas de piloto

In [ ]:
ask("¿Cuántos títulos mundiales tiene Max Verstappen?")


### 8.3 Clasificación de campeonato

In [ ]:
ask("¿Cómo va el campeonato de pilotos de F1 en 2025?")


### 8.4 Calendario

In [ ]:
ask("¿Cuál es la próxima carrera del calendario de F1 2025?")


---
### 8.5 Consulta libre
Cambia la pregunta y vuelve a ejecutar la celda.


In [ ]:
ask("¿Quién ganó las 24 Horas de Le Mans 2024?")
